- Seven fit setups on the 135 LEGA-C DR2 quiescent refit targets.
- (a) October refit reused; (d) reuses 61 carbon-mask fits.
- (f), (g): 61 Mg b galaxies refitted below Mg b; other 74 reused.
- Fit seed 20260832 everywhere. Bootstrap seed 20261110, 10000 draws.
- Age: posterior median mass-weighted age from derived outputs.
- C/N quantiles: weighted nested-sampling samples of `ceridwen_result.h5`.
- Mg b group: October fit has fitted pixels in the Mg b bandpass.
- Mg b window: [Trager et al. 1998, Table 2](https://arxiv.org/pdf/astro-ph/9712258), bandpass plus pseudo-continua, air.

In [1]:
from pathlib import Path
import json
import sys
import h5py
import jax
import numpy as np
import pandas as pd
import astropy.units as u
from astropy.cosmology import Planck18
from scipy.stats import linregress, spearmanr

jax.config.update('jax_enable_x64', True)
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts/experiment.py').exists())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'scripts'))
from scripts.experiment import validate_result
from absorption_mask_analysis import weighted_quantiles
from ceridwen.fit import load_result_h5
from ceridwen.cosmology import age_gyr, E_of_z, resolve_cosmology
from ceridwen.observation.absorption_features import feature_windows
from ceridwen.observation.filters import air2vac

OUT = ROOT / 'results/chronometer-135-2026-11-10'
REFIT = ROOT / 'results/dr2-refit-2026-10-12'
CARBON = ROOT / 'results/carbon-mask-2026-11-08'
TARGETS = [t for t in json.loads((REFIT / 'targets.json').read_text())['targets'] if 'folder' in t]
assert len(TARGETS) == 135
MGB_61 = set(json.loads((OUT / 'cn-free-noMgb/experiment.json').read_text())['targets'])
MGB_WINDOW_REST_VAC = np.asarray(air2vac(np.array([5142.625, 5206.375])))  # Trager+98 Table 2, Mg b


def run_dir(run, arm, t):
    return run / 'run/fits' / arm / f"{t['object_id']}-{t['spect_id']}"


def first(*paths):
    return next((p for p in paths if p.exists()), paths[-1])


ARMS = {
    'a_baseline': lambda t: REFIT / t['folder'],
    'b_cn_free': lambda t: run_dir(OUT / 'cn-free', 'cn_free', t),
    'c_cn_afe0p25': lambda t: run_dir(OUT / 'cn-afe0p25', 'cn_afe0p25', t),
    'd_carbon_mask': lambda t: first(run_dir(CARBON / 'check', 'carbon_mask', t), run_dir(CARBON / 'rest', 'carbon_mask', t),
                                     run_dir(OUT / 'carbon-mask', 'carbon_mask', t)),
    'e_cn_afe0p25_common': lambda t: run_dir(OUT / 'cn-afe0p25-common', 'cn_afe0p25_common', t),
    'f_cn_free_noMgb': lambda t: (run_dir(OUT / 'cn-free-noMgb', 'cn_free_noMgb', t) if t['spect_id'] in MGB_61
                                  else ARMS['b_cn_free'](t)),
    'g_baseline_noMgb': lambda t: (run_dir(OUT / 'baseline-noMgb', 'baseline_noMgb', t) if t['spect_id'] in MGB_61
                                   else ARMS['a_baseline'](t)),
}
FIXED_AFE = {'c_cn_afe0p25': 0.25, 'e_cn_afe0p25_common': 0.25}

- Validate every fit; failures stay in the table, out of statistics.
- `fit_rest_*`: first and last fitted pixel, rest vacuum Å.

In [2]:
def fit_row(arm, t):
    directory = ARMS[arm](t)
    row = dict(arm=arm, target=t['spect_id'], path=str(directory.relative_to(ROOT)))
    if not directory.exists():
        return row | dict(valid=False, failure='no result directory')
    try:
        validate_result(directory)
        row |= dict(valid=True, failure='')
    except Exception as error:
        return row | dict(valid=False, failure=str(error).split(':')[0])
    source = json.loads((directory.parents[2] / 'manifest.json').read_text())['source']
    row |= dict(astro_commit=source['commit'][:8], ceridwen_commit=source['submodules']['ceridwen'][:8])
    with h5py.File(directory / 'ceridwen_derived_outputs.h5') as f:
        names = f['summary/parameter'].asstr()[:]
        q = {k: dict(zip(names, f[f'summary/{k}'][:])) for k in ('q16', 'q50', 'q84')}
        row |= dict(z=float(f.attrs['redshift']), seed=int(f.attrs['random_seed']))
        for label, name in (('age_gyr', 'mass-weighted age [Gyr]'), ('feh', 'logzsol'),
                            ('afe', 'afe'), ('tau_dust', 'diffuse_tau_noll')):
            for k in q:
                row[f'{label}_{k}'] = float(q[k][name]) if name in q[k] else FIXED_AFE.get(arm, np.nan)
        wave = f['spectrum/wavelength'][:].astype(float)
        mask = f['spectrum/mask'][:].astype(bool)
    rest = wave / (1 + row['z'])
    lo, hi = feature_windows(['Mgb'], zred=row['z'])[0]
    row |= dict(fitted_pixels=int(mask.sum()), fit_rest_min=rest[mask].min(), fit_rest_max=rest[mask].max(),
                Mgb_n=int(np.sum(mask & (wave >= lo) & (wave <= hi))),
                Mgb_window_n=int(np.sum(mask & (rest >= MGB_WINDOW_REST_VAC[0]) & (rest <= MGB_WINDOW_REST_VAC[1]))))
    if arm in ('b_cn_free', 'c_cn_afe0p25', 'e_cn_afe0p25_common', 'f_cn_free_noMgb'):
        fit = load_result_h5(directory / 'ceridwen_result.h5')
        for p in ('cfe', 'nfe'):
            values = fit.samples.get(p)
            quantiles = weighted_quantiles(np.asarray(values), np.asarray(fit.log_weights)) if values is not None else [np.nan] * 3
            row |= dict(zip((f'{p}_q16', f'{p}_q50', f'{p}_q84'), quantiles))
    return row


fits = pd.DataFrame([fit_row(arm, t) for arm in ARMS for t in TARGETS])
base = fits[fits.arm == 'a_baseline'].set_index('target')
fits['has_Mgb'] = fits.target.map(base.Mgb_n > 0)
fits['z'] = fits.target.map(base.z)
assert set(base.index[base.Mgb_n > 0]) == MGB_61
assert (fits.seed.dropna() == 20260832).all()
cut = fits[fits.arm.isin(['f_cn_free_noMgb', 'g_baseline_noMgb']) & fits.target.isin(MGB_61) & fits.valid]
assert (cut.Mgb_window_n == 0).all() and (cut.fit_rest_max < MGB_WINDOW_REST_VAC[0]).all()
fits.to_csv(OUT / 'per-fit.csv', index=False)
validation = fits.groupby('arm').agg(fits=('valid', 'size'), valid=('valid', 'sum'))
validation['failed'] = validation.fits - validation.valid
validation['pins'] = fits.groupby('arm').apply(
    lambda g: '; '.join(f'{a}/{c}: {n}' for (a, c), n in g.groupby(['astro_commit', 'ceridwen_commit']).size().items()))
validation.to_csv(OUT / 'validation.csv')
validation

,fits,valid,failed,pins
arm,,,,
a_baseline,135,135,0,7033ef13/d7c0536f: 1; 85d3acbf/d7c0536f: 63; a...
b_cn_free,135,135,0,99c179ba/13791e0e: 135
c_cn_afe0p25,135,135,0,99c179ba/13791e0e: 135
d_carbon_mask,135,135,0,33a06784/a0b1f6ed: 61; 99c179ba/13791e0e: 74
e_cn_afe0p25_common,135,130,5,99c179ba/13791e0e: 130
f_cn_free_noMgb,135,135,0,99c179ba/13791e0e: 135
g_baseline_noMgb,135,135,0,85d3acbf/d7c0536f: 37; 99c179ba/13791e0e: 61; ...


- (e): fitted pixels of each galaxy inside 4155.0–4438.1 Å.

In [3]:
common_range = fits[fits.arm == 'e_cn_afe0p25_common'][['target', 'z', 'valid', 'fitted_pixels', 'fit_rest_min', 'fit_rest_max']]
common_range = common_range.join(base[['fitted_pixels']].rename(columns={'fitted_pixels': 'baseline_fitted_pixels'}), on='target')
common_range.to_csv(OUT / 'common-range-pixels.csv', index=False)
common_range[['fitted_pixels', 'baseline_fitted_pixels']].describe().round(1)

,fitted_pixels,baseline_fitted_pixels
count,130.0,135.0
mean,658.3,3621.6
std,61.6,99.4
min,513.0,3272.0
25%,593.0,3561.0
50%,667.0,3604.0
75%,681.0,3635.5
max,785.0,3884.0


- Common sample: galaxies valid in every arm.
- Same bootstrap draws for every arm within a group.
- Offset: Mg b minus no-Mg b median residual about one line.
- Step model: age = a + b z + c [Mg b], least squares.

In [4]:
ages = fits.pivot(index='target', columns='arm', values='age_gyr_q50')
valid = fits.pivot(index='target', columns='arm', values='valid').astype(bool).all(axis=1)
common = base.loc[valid[valid].index]
GROUPS = {'all': common.index, 'with_Mgb': common.index[common.Mgb_n > 0],
          'without_Mgb': common.index[common.Mgb_n == 0]}
COSMO = resolve_cosmology(None)
dt_dz = jax.vmap(jax.grad(lambda z: age_gyr(z, COSMO)))
H_UNIT = (1 / u.Gyr).to_value(u.km / u.s / u.Mpc)


def hubble(slope, z):
    # Jimenez & Loeb 2002 eq. 2 with dt/dz = d(age)/dz: H = -1 / ((1 + z) dt/dz)
    return -H_UNIT / ((1 + z) * slope) if slope < 0 else np.nan


def line_fit(z, y, indices):
    fit = linregress(z, y)
    zc = z[indices] - z[indices].mean(axis=1, keepdims=True)
    with np.errstate(invalid='ignore', divide='ignore'):
        draws = np.sum(zc * y[indices], axis=1) / np.sum(zc**2, axis=1)
    q16, q50, q84 = np.nanquantile(draws, [0.16, 0.5, 0.84])
    return dict(slope=fit.slope, q16=q16, q50=q50, q84=q84,
                scatter=np.std(y - fit.intercept - fit.slope * z, ddof=2))


def offset_and_step(z, y, mgb):
    line = np.polyfit(z, y, 1)
    r = y - np.polyval(line, z)
    offset = np.median(r[mgb]) - np.median(r[~mgb])
    step = np.linalg.lstsq(np.column_stack([np.ones_like(z), z, mgb.astype(float)]), y, rcond=None)[0]
    return offset, step[1], step[2]


rng = np.random.default_rng(20261110)
INDICES = {g: rng.integers(0, len(ix), size=(10000, len(ix))) for g, ix in GROUPS.items()}
valid_by_arm = fits.pivot(index='target', columns='arm', values='valid').astype(bool).loc[base.index]


def slope_row(sample, arm, group, index, indices, steps):
    z = base.loc[index, 'z'].to_numpy()
    y = ages.loc[index, arm].to_numpy()
    z_med = float(np.median(z))
    universe = line_fit(z, np.asarray(age_gyr(z, COSMO)), indices)
    s = line_fit(z, y, indices)
    sub = fits[(fits.arm == arm) & fits.target.isin(index)]
    rho = spearmanr(sub.z, sub.afe_q50) if sub.afe_q50.nunique() > 1 else (np.nan, np.nan)
    row = dict(
        sample=sample, group=group, arm=arm, galaxies=len(index), z_min=z.min(), z_max=z.max(), z_median=z_med,
        slope_gyr_per_z=s['slope'], bootstrap_q16=s['q16'], bootstrap_q50=s['q50'], bootstrap_q84=s['q84'],
        residual_scatter_gyr=s['scatter'],
        tuniv_line_slope_gyr_per_z=universe['slope'], tuniv_tangent_at_zmed_gyr_per_z=float(dt_dz(np.array([z_med]))[0]),
        H_kms_mpc=hubble(s['slope'], z_med), H_from_q16_slope=hubble(s['q16'], z_med),
        H_from_q84_slope=hubble(s['q84'], z_med), H_fit_cosmology_kms_mpc=float(COSMO.H0 * E_of_z(z_med, COSMO)),
        H_from_tuniv_line=hubble(universe['slope'], z_med),
        rho_afe_z=rho[0], rho_afe_z_p=rho[1],
        afe_median=sub.afe_q50.median(), age_median_gyr=sub.age_gyr_q50.median(),
        feh_median=sub.feh_q50.median(), cfe_median=sub.cfe_q50.median(), nfe_median=sub.nfe_q50.median())
    if steps:
        mgb = (base.loc[index, 'Mgb_n'] > 0).to_numpy()
        offset, step_slope, step = offset_and_step(z, y, mgb)
        draws = np.array([offset_and_step(z[i], y[i], mgb[i]) for i in indices])
        q = np.nanquantile(draws, [0.16, 0.84], axis=0)
        row |= dict(mgb_offset_gyr=offset, mgb_offset_q16=q[0, 0], mgb_offset_q84=q[1, 0],
                    step_model_slope_gyr_per_z=step_slope, step_model_slope_q16=q[0, 1], step_model_slope_q84=q[1, 1],
                    step_model_step_gyr=step, step_model_step_q16=q[0, 2], step_model_step_q84=q[1, 2],
                    H_step_model_kms_mpc=hubble(step_slope, z_med))
    return row


rows = [slope_row('common', arm, g, ix, INDICES[g], g == 'all') for g, ix in GROUPS.items() for arm in ARMS]
for arm in ARMS:  # each arm on its own valid fits, when that differs from the common sample
    if valid_by_arm[arm].all() and len(common) < len(base):
        ok = valid_by_arm[arm]
        for g, ix in {'all': base.index[ok], 'with_Mgb': base.index[ok & (base.Mgb_n > 0)],
                      'without_Mgb': base.index[ok & (base.Mgb_n == 0)]}.items():
            draws = np.random.default_rng(20261110).integers(0, len(ix), size=(10000, len(ix)))
            rows.append(slope_row('all_135', arm, g, ix, draws, g == 'all'))
slopes = pd.DataFrame(rows)
assert np.isclose(Planck18.H(slopes.z_median[0]).to_value('km/(s Mpc)'), slopes.H_fit_cosmology_kms_mpc[0], rtol=1e-3)
slopes.to_csv(OUT / 'age-redshift-slopes.csv', index=False)
slopes[['sample', 'group', 'arm', 'galaxies', 'slope_gyr_per_z', 'bootstrap_q16', 'bootstrap_q84', 'residual_scatter_gyr',
        'H_kms_mpc', 'rho_afe_z', 'afe_median', 'age_median_gyr', 'feh_median', 'mgb_offset_gyr',
        'step_model_slope_gyr_per_z']].round(3)

,sample,group,arm,galaxies,slope_gyr_per_z,bootstrap_q16,bootstrap_q84,residual_scatter_gyr,H_kms_mpc,rho_afe_z,afe_median,age_median_gyr,feh_median,mgb_offset_gyr,step_model_slope_gyr_per_z
0,common,all,a_baseline,130,-1.335,-2.026,-0.638,0.821,423.785,-0.488,-0.030,4.057,-0.285,0.278,-1.307
1,common,all,b_cn_free,130,1.362,0.606,2.157,0.992,NaN,-0.274,0.123,4.436,-0.418,-0.830,-1.839
2,common,all,c_cn_afe0p25,130,1.557,0.785,2.371,1.038,NaN,NaN,0.250,4.379,-0.507,-0.820,-1.646
3,common,all,d_carbon_mask,130,-2.443,-3.142,-1.734,0.824,231.663,-0.580,0.095,4.393,-0.370,0.102,-2.689
4,common,all,e_cn_afe0p25_common,130,-3.767,-4.418,-3.093,0.799,150.231,NaN,0.250,4.930,-0.434,-0.168,-4.464
5,common,all,f_cn_free_noMgb,130,-0.078,-0.795,0.658,0.941,7242.948,0.180,-0.084,4.656,-0.416,-0.208,-2.560
6,common,all,g_baseline_noMgb,130,-0.893,-1.571,-0.218,0.810,633.442,-0.304,-0.099,3.936,-0.297,0.051,-1.443
7,common,with_Mgb,a_baseline,60,-3.236,-5.640,-0.937,0.864,179.728,-0.233,0.087,4.231,-0.238,NaN,NaN
8,common,with_Mgb,b_cn_free,60,-3.541,-6.113,-0.732,0.994,164.211,-0.242,0.213,3.717,-0.281,NaN,NaN
9,common,with_Mgb,c_cn_afe0p25,60,-4.184,-6.830,-1.244,1.017,138.998,NaN,0.250,3.674,-0.290,NaN,NaN


- Paired per-galaxy age differences on fits valid in both arms.

In [5]:
PAIRS = [('b_cn_free', 'a_baseline'), ('c_cn_afe0p25', 'a_baseline'), ('d_carbon_mask', 'a_baseline'),
         ('e_cn_afe0p25_common', 'a_baseline'), ('f_cn_free_noMgb', 'a_baseline'), ('g_baseline_noMgb', 'a_baseline'),
         ('c_cn_afe0p25', 'b_cn_free'), ('e_cn_afe0p25_common', 'c_cn_afe0p25'),
         ('f_cn_free_noMgb', 'b_cn_free')]
paired = base[['z', 'Mgb_n']].copy()
rows = []
for new, old in PAIRS:
    ok = valid_by_arm[new] & valid_by_arm[old]
    paired[f'dage_{new}_minus_{old}'] = (ages[new] - ages[old]).where(ok)
    for group, index in {'all': base.index[ok], 'with_Mgb': base.index[ok & (base.Mgb_n > 0)],
                         'without_Mgb': base.index[ok & (base.Mgb_n == 0)]}.items():
        d = (ages.loc[index, new] - ages.loc[index, old]).to_numpy()
        z = base.loc[index, 'z'].to_numpy()
        s = line_fit(z, d, np.random.default_rng(20261110).integers(0, len(index), size=(10000, len(index))))
        rho = spearmanr(z, d) if np.ptp(d) > 0 else (np.nan, np.nan)
        rows.append(dict(pair=f'{new} - {old}', group=group, galaxies=len(d), dage_median=np.median(d),
                         dage_q16=np.quantile(d, 0.16), dage_q84=np.quantile(d, 0.84), older_fraction=np.mean(d > 0),
                         dage_vs_z_slope=s['slope'], dage_vs_z_q16=s['q16'], dage_vs_z_q84=s['q84'],
                         rho_dage_z=rho[0], rho_dage_z_p=rho[1]))
paired.to_csv(OUT / 'paired-age-differences-per-galaxy.csv')
paired_summary = pd.DataFrame(rows)
paired_summary.to_csv(OUT / 'paired-age-differences.csv', index=False)
paired_summary.round(3)

,pair,group,galaxies,dage_median,dage_q16,dage_q84,older_fraction,dage_vs_z_slope,dage_vs_z_q16,dage_vs_z_q84,rho_dage_z,rho_dage_z_p
0,b_cn_free - a_baseline,all,135,0.312,-0.537,1.002,0.704,2.526,2.007,3.061,0.363,0.000
1,b_cn_free - a_baseline,with_Mgb,61,-0.150,-0.920,0.483,0.443,0.007,-2.190,2.601,0.009,0.944
2,b_cn_free - a_baseline,without_Mgb,74,0.517,0.087,1.154,0.919,-0.712,-1.404,-0.040,-0.123,0.295
3,c_cn_afe0p25 - a_baseline,all,135,0.199,-0.612,0.904,0.637,2.731,2.208,3.263,0.399,0.000
4,c_cn_afe0p25 - a_baseline,with_Mgb,61,-0.177,-1.022,0.346,0.377,-0.636,-2.774,1.911,-0.055,0.673
5,c_cn_afe0p25 - a_baseline,without_Mgb,74,0.502,0.018,1.127,0.851,-0.365,-1.123,0.385,0.012,0.920
6,d_carbon_mask - a_baseline,all,135,0.250,-0.054,0.682,0.793,-1.171,-1.426,-0.907,-0.280,0.001
7,d_carbon_mask - a_baseline,with_Mgb,61,0.323,0.065,0.713,0.852,-1.990,-3.072,-1.020,-0.230,0.074
8,d_carbon_mask - a_baseline,without_Mgb,74,0.138,-0.072,0.598,0.743,-1.350,-1.713,-0.995,-0.365,0.001
9,e_cn_afe0p25_common - a_baseline,all,130,0.717,-0.164,1.509,0.792,-2.431,-3.105,-1.743,-0.247,0.005


- The 61 Mg b galaxies: Mg b cut minus full range, per galaxy.

In [6]:
QUANTITIES = ['afe_q50', 'age_gyr_q50', 'cfe_q50', 'nfe_q50']
wide = fits.set_index(['target', 'arm'])[QUANTITIES + ['z']].unstack('arm')
cut_rows, cut_summary = [], []
for new, old in [('f_cn_free_noMgb', 'b_cn_free'), ('g_baseline_noMgb', 'a_baseline')]:
    index = sorted(MGB_61 & set(base.index[valid_by_arm[new] & valid_by_arm[old]]))
    frame = pd.DataFrame({'target': index, 'pair': f'{new} - {old}', 'z': wide.loc[index, ('z', old)].to_numpy()})
    for q in QUANTITIES:
        frame[f'{q}_{old}'] = wide.loc[index, (q, old)].to_numpy()
        frame[f'{q}_{new}'] = wide.loc[index, (q, new)].to_numpy()
        frame[f'delta_{q}'] = frame[f'{q}_{new}'] - frame[f'{q}_{old}']
        d = frame[f'delta_{q}'].dropna()
        if len(d):
            cut_summary.append(dict(pair=f'{new} - {old}', quantity=q, galaxies=len(d), median=d.median(),
                                    q16=d.quantile(0.16), q84=d.quantile(0.84), positive_fraction=(d > 0).mean()))
    cut_rows.append(frame)
pd.concat(cut_rows).to_csv(OUT / 'mgb-cut-paired-per-galaxy.csv', index=False)
cut_summary = pd.DataFrame(cut_summary)
cut_summary.to_csv(OUT / 'mgb-cut-paired.csv', index=False)
cut_summary.round(3)

,pair,quantity,galaxies,median,q16,q84,positive_fraction
0,f_cn_free_noMgb - b_cn_free,afe_q50,61,-0.221,-0.392,-0.072,0.033
1,f_cn_free_noMgb - b_cn_free,age_gyr_q50,61,0.155,-0.216,0.929,0.705
2,f_cn_free_noMgb - b_cn_free,cfe_q50,61,-0.123,-0.288,-0.001,0.164
3,f_cn_free_noMgb - b_cn_free,nfe_q50,61,0.013,-0.090,0.223,0.639
4,g_baseline_noMgb - a_baseline,afe_q50,61,-0.106,-0.202,-0.015,0.049
5,g_baseline_noMgb - a_baseline,age_gyr_q50,61,-0.091,-0.500,0.134,0.344


- Pin check: same-pin baseline of the carbon-mask check against October.

In [7]:
check = []
for t in TARGETS:
    directory = run_dir(CARBON / 'check', 'baseline', t)
    if directory.exists():
        with h5py.File(directory / 'ceridwen_derived_outputs.h5') as f:
            q = dict(zip(f['summary/parameter'].asstr()[:], f['summary/q50'][:]))
        b = base.loc[t['spect_id']]
        check.append(dict(target=t['spect_id'], dage=q['mass-weighted age [Gyr]'] - b.age_gyr_q50,
                          dafe=q['afe'] - b.afe_q50, dfeh=q['logzsol'] - b.feh_q50))
pin_check = pd.DataFrame(check)
pin_check.to_csv(OUT / 'pin-check.csv', index=False)
pin_check.describe().loc[['count', 'mean', 'min', 'max']].round(4)

,dage,dafe,dfeh
count,10.0000,10.0000,10.0000
mean,0.0371,-0.0010,-0.0027
min,-0.0520,-0.0128,-0.0172
max,0.1487,0.0064,0.0036
